<a href="https://colab.research.google.com/github/tardigrade-dot/colab-script/blob/main/03_generate_srt_funasr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#funasr所需要的numpy版本与当前colab的不兼容, 需要降低colab的python版本
#!pip install -r https://raw.githubusercontent.com/FunAudioLLM/Fun-ASR/refs/heads/main/requirements.txt
!pip install -r https://raw.githubusercontent.com/FunAudioLLM/SenseVoice/refs/heads/main/requirements.txt
!pip install wetext


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    # print(f"PyTorch CUDA version: {torch.backends.cuda.version()}")

# Uninstall and reinstall torch, torchvision, and torchaudio to ensure CUDA compatibility
# This often fixes CUDA version mismatches in Colab environments.
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!git clone https://github.com/FunAudioLLM/Fun-ASR
import os
os.chdir('/content/Fun-ASR')
current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")


In [ ]:
!pip install huggingface_hub

In [ ]:
# Re-install numpy==1.26.4 as the last step to ensure it's the active version
!pip uninstall numpy -y
!pip install numpy==1.26.4

# Verify numpy version immediately after installation
import numpy as np
print(f"Numpy version (after final re-installation): {np.__version__}")
print(f"Numpy path (after final re-installation): {np.__path__}")

# IMPORTANT: After running this cell, please restart the Colab runtime (Runtime -> Restart runtime...)
# and then run all cells from the beginning to ensure changes are fully applied.

In [ ]:
# !hf download FunAudioLLM/Fun-ASR-Nano-2512 --local-dir /content/Fun-ASR-Nano-2512

In [ ]:
!hf download FunAudioLLM/SenseVoiceSmall --local-dir /content/SenseVoiceSmall

In [ ]:
import sys
import os

current_dir = os.getcwd()
directory_to_add = '/content/Fun-ASR'
if directory_to_add not in sys.path:
    sys.path.insert(0, directory_to_add) # insert at the beginning to give it priority
    print(f"Added '{directory_to_add}' to sys.path")


In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4

In [ ]:
import numpy as np
print(f"Numpy version: {np.__version__}")
print(f"Numpy path: {np.__path__}")

In [ ]:
import re
import os
import string
import difflib
from pathlib import Path
from funasr import AutoModel
from pydub import AudioSegment
from wetext import Normalizer

class CtcForcdAlign:
    def __init__(self, model_dir, device) -> None:

        self.MIN_CHARS_PER_LINE = 10
        self.MAX_MISMATCH_RATIO_ABORT = 0.20  # 整体不匹配率超过此值则放弃生成字幕
        self.MAX_GAP_NONE_CHARS = 5           # 允许连续无时间戳的最大字符数（用于插值容忍）
        self.model = AutoModel(
            model=model_dir,
            vad_model="fsmn-vad",
            vad_kwargs={"max_single_segment_time": 30_000},
            device=device,
            disable_update=True,
            trust_remote_code=True,
            remote_code="funasr.models.sense_voice.model",
            attn_implementation="flash_attention_2",
            # attn_implementation="sdpa",
        )

        self.normalizer = Normalizer(lang="zh", operator="tn", remove_erhua=True, traditional_to_simple=True)
        chinese_pattern = r"（.*?）"
        english_pattern = r"\([^)]*?\)"

        self.combined_pattern = f"{chinese_pattern}|{english_pattern}"
        self.PUNCTUATION_CHARS = set(string.punctuation) | set(',.?，。！？；：、""''《》【】（）')
        self.punctuation_regex = r'[.,:;!?，。、；：？！""''·（）「」《》【】\s-]'
        self.PUNCT_REGEX = re.compile(self.punctuation_regex)

    def is_punctuation(self, token):
        return token in self.PUNCTUATION_CHARS

    def map_opcodes_to_raw(self, clean_opcodes, asr_timestamps, correct_tokens_raw, debug=False):
        aligned_timestamps = []
        status = []

        j_raw_ptr = 0

        # 用于诊断：记录每个 opcode 段的信息
        if debug:
            opcode_summary = []

        for tag, i1, i2, j1, j2 in clean_opcodes:

            while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
                aligned_timestamps.append((None, None))
                j_raw_ptr += 1

            asr_chars_consumed = i2 - i1
            if tag == "equal":
                for k in range(j2 - j1): # 遍历匹配的汉字
                    aligned_timestamps.append(asr_timestamps[i1 + k])
                    status.append(True)
                    j_raw_ptr += 1 # 原始指针移动一个汉字
                    while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
                        aligned_timestamps.append((None, None))
                        j_raw_ptr += 1
            elif tag == "replace" or tag == "insert":
                for k in range(j2 - j1): # 遍历差异的汉字
                    if asr_chars_consumed > 0:
                        idx = i1 + k % asr_chars_consumed
                        aligned_timestamps.append(asr_timestamps[idx])
                    else: # 纯粹的插入 (tag='insert')
                        aligned_timestamps.append((None, None))
                    status.append(False)
                    j_raw_ptr += 1 # 原始指针移动一个汉字
                    while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
                        aligned_timestamps.append((None, None))
                        j_raw_ptr += 1
            elif tag == "delete":
                continue

            if debug:
                opcode_summary.append(f"  {tag}: ASR[{i1}:{i2}] -> Text[{j1}:{j2}] ({i2-i1} -> {j2-j1} chars)")

        while j_raw_ptr < len(correct_tokens_raw) and self.PUNCT_REGEX.match(correct_tokens_raw[j_raw_ptr]):
            aligned_timestamps.append((None, None))
            j_raw_ptr += 1

        if debug:
            print(f"\n========== 对齐诊断 ==========")
            print(f"ASR tokens ({len(asr_timestamps)}): {''.join(asr_timestamps[i][2] if len(asr_timestamps[i]) > 2 else '?' for i in range(len(asr_timestamps)))}")
            print(f"Correct text ({len(correct_tokens_raw)}): {correct_tokens_raw}")
            print(f"Opcodes:")
            for s in opcode_summary:
                print(s)
            match_count = status.count(True)
            mismatch_count = status.count(False)
            print(f"匹配字符: {match_count}, 不匹配/插入字符: {mismatch_count}")

            # 显示连续不匹配的区域
            none_runs = []
            run_start = None
            for idx, ts in enumerate(aligned_timestamps):
                if ts == (None, None):
                    if run_start is None:
                        run_start = idx
                else:
                    if run_start is not None:
                        none_runs.append((run_start, idx - 1))
                        run_start = None
            if run_start is not None:
                none_runs.append((run_start, len(aligned_timestamps) - 1))

            if none_runs:
                print(f"\n连续无时间戳区域 (共 {len(none_runs)} 段):")
                for start, end in none_runs:
                    chars = correct_tokens_raw[start:end+1]
                    print(f"  [{start}-{end}] ({end-start+1} chars): '{''.join(chars)}'")
            print(f"============================\n")

        if len(aligned_timestamps) == len(correct_tokens_raw):
            return aligned_timestamps, status
        else:
            print(f"🚨 警告: 最终长度不匹配. Raw: {len(correct_tokens_raw)}, Aligned: {len(aligned_timestamps)}")
            return aligned_timestamps, status

    def map_asr_to_correct(self, asr_tokens, asr_timestamps, correct_tokens, debug=False):

        clean_correct_tokens = re.sub(self.punctuation_regex, '', correct_tokens)
        matcher_clean = difflib.SequenceMatcher(None, asr_tokens, clean_correct_tokens)
        clean_opcodes = matcher_clean.get_opcodes()

        if debug:
            print(f"\n[DIAG] ASR识别结果 ({len(asr_tokens)}字): {''.join(asr_tokens)}")
            print(f"[DIAG] 标准文本 ({len(clean_correct_tokens)}字): {clean_correct_tokens}")
            print(f"[DIAG] difflib opcodes 数量: {len(clean_opcodes)}")

        return self.map_opcodes_to_raw(clean_opcodes, asr_timestamps, correct_tokens, debug=debug)

    def is_valid_time(self, t):
        return isinstance(t, (int, float)) and t >= 0

    def get_valid_start_end(self, words):
        start_time, end_time = None, None

        for w in words:
            t = w.get('start')
            if self.is_valid_time(t):
                start_time = t
                break

        for w in reversed(words):
            t = w.get('end')
            if self.is_valid_time(t):
                end_time = t
                break

        return start_time, end_time

    def fill_none_timestamps(self, words):
        """
        对 words 列表中 start/end 为 None 的位置进行线性插值填补。
        策略：找到头部和尾部的有效时间戳作为锚点，中间缺失的部分均匀分配。
        如果无法找到至少两个锚点，返回 False。
        """
        valid_indices = []
        for i, w in enumerate(words):
            s, e = w.get('start'), w.get('end')
            if self.is_valid_time(s) and self.is_valid_time(e):
                valid_indices.append((i, s, e))

        if len(valid_indices) < 2:
            # 只有一个锚点或没有：无法进行可靠插值
            if len(valid_indices) == 1:
                idx, s, e = valid_indices[0]
                char_dur = max(e - s, 200)  # 单个字的时长下限 200ms
                for i in range(len(words)):
                    if words[i]['start'] is None or words[i]['end'] is None:
                        offset = i - idx
                        words[i]['start'] = max(0, s + offset * char_dur)
                        words[i]['end'] = max(0, e + offset * char_dur)
                return True
            return False

        first_idx, first_s, first_e = valid_indices[0]
        last_idx, last_s, last_e = valid_indices[-1]

        span_chars = last_idx - first_idx
        span_duration = last_s - first_s

        if span_chars > 0 and span_duration > 0:
            avg_dur_per_char = span_duration / span_chars
        else:
            char_dur = first_e - first_s
            avg_dur_per_char = max(char_dur, 200)

        # 填补第一个锚点之前的部分（向前扩展）
        for i in range(first_idx - 1, -1, -1):
            if words[i]['start'] is None or words[i]['end'] is None:
                offset = first_idx - i
                words[i]['end'] = max(0, first_s - (offset - 1) * avg_dur_per_char)
                words[i]['start'] = max(0, first_s - offset * avg_dur_per_char)

        # 填补两个锚点之间的部分（线性插值）
        for i in range(first_idx + 1, last_idx):
            if words[i]['start'] is None or words[i]['end'] is None:
                offset = i - first_idx
                words[i]['start'] = first_s + offset * avg_dur_per_char
                words[i]['end'] = first_s + (offset + 1) * avg_dur_per_char

        # 填补充锚点之后的部分（向后扩展）
        for i in range(last_idx + 1, len(words)):
            if words[i]['start'] is None or words[i]['end'] is None:
                offset = i - last_idx
                words[i]['start'] = last_s + offset * avg_dur_per_char
                words[i]['end'] = last_e + offset * avg_dur_per_char

        return True

    def ms_to_srt_time_format(self, ms):
        ms = int(ms)
        hours = ms // 3600000
        minutes = (ms % 3600000) // 60000
        seconds = (ms % 60000) // 1000
        milliseconds = ms % 1000
        return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

    def generate_srt_from_words_and_timestamps(self, words, timestamps):
        if not words or not timestamps or len(words) != len(timestamps):
            print(f"words length[{len(words)}] not equal to timestamps length[{len(timestamps)}]")
            raise Exception('process error')

        srt_content = ""
        subtitle_index = 1
        sentence_delimiters = ['。', '？', '！', '.', '?', '!', '…']

        current_subtitle_words = []
        end_time = 0
        skipped_segments = []

        def _has_valid_anchors(words_list):
            """检查是否有至少两个有效锚点（可用于插值）"""
            count = 0
            for w in words_list:
                if self.is_valid_time(w.get('start')) and self.is_valid_time(w.get('end')):
                    count += 1
                    if count >= 2:
                        return True
            return count >= 1  # 至少一个也能勉强处理

        def _write_subtitle(words_to_write, pre_end_time):
            nonlocal srt_content, subtitle_index, skipped_segments

            if not words_to_write:
                return None

            # 第一步：尝试插值填补 None 时间戳
            if not _has_valid_anchors(words_to_write):
                text = "".join([w['word'] for w in words_to_write])
                skipped_segments.append(text)
                # 诊断信息：统计有多少个 None
                none_count = sum(1 for w in words_to_write if w['start'] is None)
                total = len(words_to_write)
                print(f"⚠️ 跳过字幕段（{none_count}/{total} 字无时间戳，无锚点可插值）: {text}")
                return pre_end_time

            # 有锚点，尝试插值
            self.fill_none_timestamps(words_to_write)

            # 第二步：获取起止时间
            start_time, end_time = self.get_valid_start_end(words_to_write)

            # 如果仍然无法获取有效时间，跳过
            if not self.is_valid_time(start_time) or not self.is_valid_time(end_time):
                text = "".join([w['word'] for w in words_to_write])
                skipped_segments.append(text)
                print(f"⚠️ 跳过字幕段（插值后仍无有效时间戳）: {text}")
                return pre_end_time

            # 确保 start_time 不早于上一段的 end_time
            if pre_end_time and pre_end_time != 0 and start_time < pre_end_time:
                start_time = pre_end_time

            sentence_text = "".join([w['word'] for w in words_to_write])

            srt_content += str(subtitle_index) + "\n"
            srt_content += f"{self.ms_to_srt_time_format(start_time)} --> {self.ms_to_srt_time_format(end_time)}\n"
            srt_content += sentence_text + "\n\n"
            subtitle_index += 1
            return end_time

        for i in range(len(words)):
            word = words[i]
            timestamp = timestamps[i]

            current_subtitle_words.append({'word': word, 'start': timestamp[0], 'end': timestamp[1]})

            is_sentence_end = word in sentence_delimiters
            if is_sentence_end:
                current_text = "".join([w['word'] for w in current_subtitle_words])
                current_length = len(current_text)
                if current_length >= self.MIN_CHARS_PER_LINE:
                    end_time = _write_subtitle(current_subtitle_words, end_time)
                    current_subtitle_words = []

        # 处理剩余不足一行的部分
        if current_subtitle_words:
            end_time = _write_subtitle(current_subtitle_words, end_time)

        if skipped_segments:
            print(f"⚠️ 共有 {len(skipped_segments)} 段因无法找到时间锚点而被跳过: {skipped_segments}")

        if not srt_content:
            raise Exception("未生成任何字幕内容，所有段落均被跳过。请检查音频质量。")

        return srt_content

    def generate_srt_file(self, wav_file, over_write=False, debug=False):

        print(f'generate srt file for wav {wav_file}')
        wav_path = Path(wav_file)

        txt_path = wav_path.with_suffix('.txt')
        srt_path = wav_path.with_suffix('.srt')

        txt_file = str(txt_path)
        srt_file = str(srt_path)

        print(f'\nwav [{wav_file}], \ntxt [{txt_file}], \nsrt [{srt_file}]')

        if not wav_path.exists():
            raise Exception(f'wav file[{wav_file}] not exists ')
        if not txt_path.exists():
            raise Exception(f'txt file[{txt_file}] not exists ')
        if srt_path.exists() and not over_write:
            print(f'srt file[{srt_file}] exists, just return ')
            return

        with open(txt_file, 'r') as f:
            target_txt = "".join(f.readlines())

        res = self.model.generate(
                input=wav_file,
                cache={},
                language="auto",  # "zn", "en", "yue", "ja", "ko", "nospeech"
                use_itn=False,
                batch_size_s=30,
                merge_vad=True,
                merge_length_s=20,
                output_timestamp=True,
            )
        output = res[0]
        print("output is : ", "".join(output["words"]))
        target_txt = re.sub(self.combined_pattern, "", target_txt)
        target_txt = target_txt.replace("\n", " ")
        target_txt = self.normalizer.normalize(target_txt)

        aligned_timestamps, status = self.map_asr_to_correct(output["words"], output["timestamp"], target_txt, debug=debug)

        num_tokens = len(status)
        num_mismatch = status.count(False)
        mismatch_ratio = num_mismatch / num_tokens if num_tokens > 0 else 1.0

        if mismatch_ratio > self.MAX_MISMATCH_RATIO_ABORT:
            raise Exception(
                f"⚠️ ❌ 不匹配率过高 ({mismatch_ratio:.2%}), 跳过此文件。"
                f"请检查音频质量或文本与音频的一致性。"
                f"不匹配 token({num_mismatch}/{num_tokens})"
            )
        elif mismatch_ratio > 0.05:
            print(f"⚠️ 注意, 不匹配 token({num_mismatch}/{num_tokens}) 占比: {mismatch_ratio:.2%}")
        else:
            print(f"✅ 不匹配 token({num_mismatch}/{num_tokens}) 占比: {mismatch_ratio:.2%}")

        srt_result = self.generate_srt_from_words_and_timestamps(target_txt, aligned_timestamps)

        with open(srt_file, 'w') as f:
            f.write(srt_result)

        print(f"字幕已生成, 保存在:{srt_file}")

    def get_wav_list_sorted(self, wav_src_dir, wav_regex=None):

        if wav_regex is None:
            wav_regex = r'.*-(\d+)_(\d+)\.wav$'
        pattern = re.compile(wav_regex)

        files = [
            f for f in os.listdir(wav_src_dir)
            if f.endswith('.wav') and pattern.match(f)
        ]

        def sort_key(filename):
            m = pattern.match(filename)
            if m:
                second, first = map(int, m.groups())
                return (second, first)
            return (float('inf'), float('inf'))

        sorted_files = sorted(files, key=sort_key)

        sorted_paths = [os.path.join(wav_src_dir, f) for f in sorted_files]
        return sorted_paths

    def check_srt_exsis(self, wav_src_dir):
        srt_result = []
        for w_path in self.get_wav_list_sorted(wav_src_dir):

            srt_path = Path(w_path).with_suffix('.srt')

            if not srt_path.exists():
                srt_result.append(srt_path.stem)

        print(f'以下文件没有srt文件, 请检查语音文件 {srt_result}')

    def generate_srt_dir(self, wav_src_dir, over_write=False, debug=False):
        for w_path in self.get_wav_list_sorted(wav_src_dir):

            try:
                self.generate_srt_file(w_path, over_write, debug=debug)
            except Exception as e:
                print(f"处理文件 {w_path} 时发生错误：{e}")

book_name = "wenhuaquanliyuguojia"
book_dir = f"/content/drive/MyDrive/{book_name}"
cfa = CtcForcdAlign(
    model_dir = "/content/SenseVoiceSmall",
    device = "cuda"
)

# 正常批量处理模式
# cfa.generate_srt_dir(book_dir)

# 调试单个文件：查看详细的对齐诊断信息
cfa.generate_srt_file(f"{book_dir}/wenhuaquanliyuguojia-2_3.wav", debug=True)

# cfa.generate_srt_dir(f"/Volumes/sw/tts_result/{book_name}", over_write=False)
cfa.check_srt_exsis(book_dir)

# cfa.generate_srt_file("/Volumes/sw/tts_result/sulianjianshi/sulianjianshi-4_2.wav", True)
# cfa.generate_srt_dir('/Volumes/sw/MyDrive/zhengzhi1/output', r"zhengzhi1-(\d+)_(\d+)\.wav", True)